# Private Groq live verification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/verification/groq_live_check.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

1. Open this notebook in Google Colab and run its first code cell to prepare the course files automatically.
2. In the left **Secrets** panel (key icon), create `GROQ_API_KEY`, paste the key there and enable notebook access. Do not paste it into a code cell or this conversation.
3. Set `RUN_LIVE=True` below and run all cells. The example model is listed in Groq's model documentation as of 4 October 2026; your account must have access. You may change the exact model ID.
4. Download `groq-live-report.json` at the end and share that report. It contains fictional task inputs, actual answers and telemetry; it excludes the credential. Review it before sharing.

This performs up to 30 model requests, with no automatic retry. Charges/rate limits follow your account. It checks six typed requests, native tool use and nine policy cases. Human semantic review and embeddings are separate; this report does not auto-certify production readiness.

Official references: [Groq models](https://console.groq.com/docs/models), [local tool calling](https://console.groq.com/docs/tool-use/local-tool-calling), [Google's Colab Secrets example](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb).

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
RUN_LIVE=False
MODEL='openai/gpt-oss-20b'
print('Live execution:',RUN_LIVE)


In [ ]:
if RUN_LIVE:
    try:
        from google.colab import userdata
    except ImportError:
        import getpass
        key=getpass.getpass('Groq key (hidden; not saved): ')
    else:
        key=userdata.get('GROQ_API_KEY')
    from verification.groq_check import run
    report_path,summary=run(key,MODEL)
    del key
    print(summary)
    print('Report:',report_path)
else:
    print('No API call made. Set RUN_LIVE=True after configuring the secret.')

In [ ]:
if RUN_LIVE:
    try:
        from google.colab import files
    except ImportError:
        print('Open the report in outputs/ and share it after review.')
    else:
        files.download(str(report_path))

## Interpreting the result

A failed extraction is evidence to investigate, not a reason to change the expected answer silently. A native tool pass means the correct authoritative tool total was observed; inspect the final prose separately. Policy answers have `human_correct`, `human_supported` and `human_complete` left unset for review. Provider failures are recorded without printing credentials or raw HTTP error bodies.

This notebook does not run an embedding endpoint. The main preflight/data-plane notebook covers the separately configured embedding service. Saving a secret in your Colab account does not grant this chat access to it; sharing the result file is the intended workflow.